# Alignment Walls — Research Cockpit

A hands-on workspace to **assess a set and understand *why* it fails**. Run from the **repo root** with the `venvs/audio` kernel.

The two walls (see `README.md` for the framing + dead-ends):
- **Placement** — where in the mix. Doesn't generalize across sets.
- **Structure** — *which* repeat of a chorus. ~half are physically unwinnable ("the ceiling").

The loop: **load a set → see what failed and why → click a span → listen and judge with your ears.**

> Change `SET` below to `"bb11"`, `"bb12"`, or a raw set_id.

In [ ]:
%load_ext autoreload
%autoreload 2
from eda.alignment.walls import wall_lab as w
import pandas as pd
from IPython.display import Audio, display
pd.set_option("display.max_rows", 200); pd.set_option("display.width", 160)

SET = "bb12"          # <- your set
gt = w.load_gt(SET)
print("set:", SET, "->", w.resolve_set_id(SET))
print("aligning dir:", w.aligning_dir(SET))
print("GT tracks:", len(gt.tracks), "| predicted timeline on disk?", w.have_timeline(SET))

## 1. Set X-ray — where does it fail?

Per-span predicted-vs-GT scorecard, worst-first, with a transparent **binding cause** per span
(`identity` / `placement` / `structure/decode` / `ok`). **Needs a predicted timeline on disk** —
if it's missing, run the printed command once (a few minutes), then re-run this cell.

In [ ]:
if w.have_timeline(SET):
    xray = w.set_xray(SET)
    print("failures by binding cause:")
    display(xray["cause"].value_counts())
    display(xray.head(40))
else:
    print("No predicted timeline yet. Generate it once from the repo root:\n")
    print("  " + w.ensure_timeline_cmd(SET))

## 2. Structure wall — is the repeat even *winnable*?

For each repeated span, slide the **played** content over the reference and measure the
**best-vs-runner-up margin**:
- **`ambiguous`** (tiny margin) = two reference positions look identical → **the ceiling**, no algorithm can win from audio alone.
- **`recoverable`** = the true position stands out → a beatable miss.

Needs only GT + audio (no timeline). Acappellas auto-route to **HuBERT** (key-invariant), else **chroma**.
First run loads HuBERT and is slow — `limit=` caps it for a quick pass. First-order (native tempo).

In [ ]:
struct = w.structure_report(SET, limit=25)   # drop limit for the full set (slow)
n = len(struct)
if n:
    amb = int((struct["verdict"] == "ambiguous").sum())
    print(f"{amb}/{n} repeat-ish spans are AMBIGUOUS (physical ceiling) = {amb/n:.0%}")
    rec = struct[struct.verdict == "recoverable"]
    if len(rec):
        print(f"of the 'recoverable' spans, top peak lands on GT: {rec['best_is_gt'].mean():.0%}")
display(struct)

## 3. Listen for yourself

Pick a `SLOT` from the tables above. This renders **3 clips**: the **PLAYED** span and the **two competing
reference positions**. If you can't tell the two reference clips apart, that span is *the ceiling made audible*.

In [ ]:
SLOT = struct.iloc[0]["slot"] if len(struct) else None   # default: most-ambiguous span. change me.
track = next((t for t in gt.tracks if t.slot_label == SLOT), None)
assert track is not None, f"slot {SLOT!r} not found in GT"
d = w.span_distinguishability(SET, track)
print(f"slot {SLOT} stem={track.claimed_stem} feature={d.feature if d else '?'}")
if d: print(f"  margin={d.margin:.3f} -> {d.verdict} | best_ref={d.best_ref_s:.0f}s runnerup={d.runnerup_ref_s:.0f}s gt={d.gt_ref_s:.0f}s")
clips = w.render_span_snippets(SET, track, out_dir=f"/tmp/walls_snippets/{w.resolve_set_id(SET)}")
for name, path in (clips or {}).items():
    print("\n" + name); display(Audio(filename=str(path)))

## 4. Scratch — your experiments

`w` (wall_lab), `gt`, `struct` are in scope. Starting points:
- force a feature: `w.span_distinguishability(SET, track, feature="chroma")`
- include every span, not just repeats: `w.structure_report(SET, only_repeatish=False, limit=30)`
- stricter ceiling: `w.structure_report(SET, margin_thresh=0.15, limit=25)`
- placement spread (needs timeline): `w.set_xray(SET)["place_err_s"].describe()`
- one span deep: `t = next(t for t in gt.tracks if t.slot_label=="002"); w.span_distinguishability(SET, t)`

In [ ]:
# your experiments here
